# 01 Carga y Validacion de Parquet

Este notebook consulta el `OLAP` de Victor, reconstruye la base por ticket y genera el `Parquet` principal de la etapa `01`.


## 1. Librerias


In [ ]:
from pathlib import Path
import io
import os
import warnings

import pandas as pd
import psycopg2


## 2. Definir rutas y conexion


In [ ]:
from pathlib import Path

def resolve_project_root() -> Path:
    # Buscar la raiz del proyecto a partir del directorio actual.
    current = Path.cwd().resolve()
    for candidate in [current, *current.parents]:
        if (candidate / "README.md").exists() and (candidate / "parquets").exists():
            return candidate
    raise FileNotFoundError("No se pudo localizar la raiz del proyecto")

PROJECT_ROOT = resolve_project_root()

# Definir la ruta del SQL base y la salida principal de la etapa 01.
SQL_PATH = PROJECT_ROOT / "sql" / "04_base_tickets_modelado.sql"
OUTPUT_DIR = PROJECT_ROOT / "parquets" / "01_Carga_y_Validacion_Parquet"
OUTPUT_PATH = OUTPUT_DIR / "01_base_tickets_modelado.parquet"

# Definir la conexion local a PostgreSQL.
CONNECTION_PARAMS = {
    "host": os.getenv("PGHOST", "localhost"),
    "port": os.getenv("PGPORT", "5432"),
    "dbname": os.getenv("PGDATABASE", "restaurante"),
    "user": os.getenv("PGUSER", "postgres"),
    "password": os.getenv("PGPASSWORD", "postgres"),
}

print(f"Raiz del proyecto: {PROJECT_ROOT}")
print(f"SQL de entrada: {SQL_PATH}")
print(f"Parquet de salida: {OUTPUT_PATH}")


## 3. Cargar la consulta y leer la base desde OLAP


In [ ]:
# Leer la consulta que reconstruye la base de tickets.
query = SQL_PATH.read_text(encoding="utf-8")

# Ejecutar la consulta en PostgreSQL y cargar el resultado en un DataFrame.
connection = psycopg2.connect(**CONNECTION_PARAMS)
try:
    with warnings.catch_warnings():
        warnings.filterwarnings("ignore", message="pandas only supports SQLAlchemy connectable")
        df = pd.read_sql_query(query, connection)
finally:
    connection.close()

# Mostrar las primeras filas de la base reconstruida.
df.head()


### Resultado breve

En este punto ya se tiene la base reconstruida a nivel ticket a partir del esquema `olap`. La salida todavia esta en memoria y falta validarla antes de exportarla.


## 4. Validar dimensiones y tipos


In [ ]:
# Revisar la cantidad de filas y columnas generadas.
df.shape


In [ ]:
# Revisar tipos de datos de forma compacta.
buffer = io.StringIO()
df.info(buf=buffer)
print(buffer.getvalue())


### Resultado breve

La validacion de dimensiones y tipos permite confirmar que la estructura general del ticket reconstruido es consistente antes de guardarla.


## 5. Revisar nulos y columnas clave


In [ ]:
# Revisar cuantos valores nulos existen por columna.
df.isna().sum().sort_values(ascending=False).head(15)


In [ ]:
# Revisar rapidamente las columnas clave del negocio.
df[["fecha", "ciudad", "metodo_pago", "total_pedido", "monto_pago", "ticket_alto"]].head(10)


### Resultado breve

Esta revision ayuda a detectar faltantes y a confirmar que las variables principales del negocio ya quedaron listas para la base de modelado.


## 6. Exportar el parquet de la etapa 01


In [ ]:
# Crear la carpeta de salida si todavia no existe.
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# Exportar la base principal del proyecto a formato Parquet.
df.to_parquet(OUTPUT_PATH, index=False)

# Confirmar la ruta de salida generada.
print(f"Parquet generado en: {OUTPUT_PATH}")


In [ ]:
# Volver a leer el parquet exportado para verificar que se guardo correctamente.
df_exportado = pd.read_parquet(OUTPUT_PATH)
df_exportado.shape


### Resultado breve

La etapa `01` queda cerrada cuando el archivo `01_base_tickets_modelado.parquet` ya existe, se puede leer sin errores y conserva la misma estructura de la base reconstruida.
